## threshold judgment

This notebook introduces `asserted_solution`; after running it you can record a judgment that a candidate design satisfies a requirement, following Hawkins §3.3.

The previous two notebooks established the formal structure: a requirement usage (`timely`) applied to two candidates, and a calculation (`DeliveredEnergy`) that quantifies the nominal design. Before claiming the nominal design satisfies `TimelyToast`, we need to record why that claim is appropriate and what evidence supports it. That record is an `asserted_solution` — the third Hawkins judgment type, used when evidence directly supports a conclusion.

In [ ]:
import opensysml
from toaster.report import format_diagnostics
from toaster.evidence import ReviewRecord, hash_content, validate_record

source = """package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
}"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

In [ ]:
# Negative control: a requirement usage referencing an undefined requirement def
# raises "unresolved reference" at the usage site.
bad_source = """
package Bad {
    private import ScalarValues::*;
    requirement timely_bad : UndefinedRequirement;
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
solution_record = ReviewRecord(
    identifier="AS-C03",
    kind="asserted_solution",
    claim="The nominal design (cycleTime = 120 s) satisfies TimelyToast (cycleTime <= 180 s).",
    model_ref="ToasterDemo::nominal",
    content_hash=hash_content(source),
    scope="ToasterDemo",
    criteria="TimelyToast: toaster.cycleTime <= 180.0",
    premises=[],
    assumption_refs=["AC-001"],
    evidence_refs=["assert satisfy timely by nominal"],
    rationale="120 s < 180 s; the nominal variant is within the bound by a 60 s margin.",
    counterevidence="The slow variant (200 s) violates the bound. The nominal holds only for the default cycleTime.",
    residual_uncertainties="Thermal cycling effects on actual cycle duration are not modeled.",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(solution_record)
print(f"Validation errors: {errors}")
print(f"Record kind:       {solution_record.kind}")
conn.close()

The Hawkins §3.3 schema specifies what an `asserted_solution` record must contain (A-F); filling and validating the `ReviewRecord` in Python enacts that schema (O-S); `validate_record()` returning `[]` confirms all required fields are present (E).

Try the chapter exercise in `exercises/ch03/exercise.ipynb`: write an `asserted_solution` record for your `TemperatureReq` satisfaction claim.